# Laboratorio 3. Seleccion de arquitecturas y plan de procesamiento

Fabian Prado Dluzniewski 23427

Abby Donis 22440

Hansel Lopez 19026

Este notebook define las arquitecturas que se van a entrenar en la entrega
final y el plan de procesamiento de imagenes. Corresponde a la parte del avance
que pide seleccion de modelos y plan, por lo que aqui se justifican y se
verifican las arquitecturas, no se hace el tuneo completo.

Requiere haber corrido `01_EDA.ipynb` y `02_Preprocesamiento.ipynb`, que dejan
los tensores en `data/procesado/`.

### IMPORTS

In [1]:
import time

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
np.random.seed(42)

DISPOSITIVO = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"dispositivo: {DISPOSITIVO}")

dispositivo: mps


### CARGA DE LOS DATOS PREPROCESADOS

In [2]:
from pathlib import Path

DIR_PROC = Path("./data/procesado")
norma = np.load(DIR_PROC / "normalizacion.npz", allow_pickle=True)
MEDIA, DESV = norma["media"], norma["desv"]
CLASES = list(norma["clases"])
LADO = int(norma["lado"])
N_CLASES = len(CLASES)


def cargar(grupo):
    d = np.load(DIR_PROC / f"{grupo}.npz")
    X = torch.from_numpy(d["X"]).permute(0, 3, 1, 2).float().div_(255.0)
    X = (X - torch.tensor(MEDIA).view(3, 1, 1)) / torch.tensor(DESV).view(3, 1, 1)
    return TensorDataset(X, torch.from_numpy(d["y"]))


conjuntos = {g: cargar(g) for g in ["train", "val", "test"]}
for g, ds in conjuntos.items():
    print(f"{g:6} {len(ds):,} imagenes")
print(f"clases {N_CLASES}   entrada {3}x{LADO}x{LADO}")

train  12,180 imagenes
val    2,610 imagenes
test   2,610 imagenes
clases 29   entrada 3x64x64


## 4.1. Criterios de seleccion

Las arquitecturas se eligen a partir de lo que mostro el analisis exploratorio,
no de una lista generica.

El dataset esta balanceado y tiene fondo constante, asi que el problema no es
de datos escasos ni de clases raras. La dificultad esta en distinguir clases
que comparten silueta, sobre todo el bloque U, V y R, donde la diferencia son
unos pocos pixeles entre dedos. Eso pide filtros que capten detalle local fino,
es decir convoluciones pequenas de 3x3 y suficiente profundidad para componer
bordes en formas.

Al mismo tiempo, la redundancia entre cuadros consecutivos implica que el
numero efectivo de ejemplos distintos es mucho menor que 12,180. Un modelo con
demasiados parametros memoriza la sesion de grabacion en lugar de aprender la
mano, asi que la regularizacion importa mas que el tamano.

Se comparan cuatro familias.

## 4.2. Arquitecturas de red convolucional

**CNN A, linea base.** Tres bloques convolucionales con 32, 64 y 128 filtros,
cada uno seguido de agrupamiento maximo. Es la arquitectura minima capaz de
llegar a un campo receptivo que cubra la mano completa. Sirve de referencia
para medir si la complejidad adicional de la segunda red se justifica.

**CNN B, profunda y regularizada.** Cuatro bloques con normalizacion por lotes
y descarte, y agrupamiento promedio global en lugar de aplanar. La
normalizacion por lotes estabiliza el entrenamiento y permite tasas de
aprendizaje mayores. El agrupamiento promedio global reduce muchisimo los
parametros de la capa final, que es justo donde una red de este tipo tiende a
memorizar.

In [3]:
class CNNBase(nn.Module):
    def __init__(self, n_clases=N_CLASES, filtros=(32, 64, 128), p_descarte=0.25):
        super().__init__()
        capas, entrada = [], 3
        for f in filtros:
            capas += [nn.Conv2d(entrada, f, 3, padding=1), nn.ReLU(),
                      nn.MaxPool2d(2)]
            entrada = f
        self.rasgos = nn.Sequential(*capas)
        lado_final = LADO // (2 ** len(filtros))
        self.clasificador = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p_descarte),
            nn.Linear(entrada * lado_final * lado_final, 256), nn.ReLU(),
            nn.Dropout(p_descarte),
            nn.Linear(256, n_clases),
        )

    def forward(self, x):
        return self.clasificador(self.rasgos(x))


class CNNProfunda(nn.Module):
    def __init__(self, n_clases=N_CLASES, filtros=(32, 64, 128, 256), p_descarte=0.3):
        super().__init__()
        capas, entrada = [], 3
        for f in filtros:
            capas += [nn.Conv2d(entrada, f, 3, padding=1), nn.BatchNorm2d(f), nn.ReLU(),
                      nn.Conv2d(f, f, 3, padding=1), nn.BatchNorm2d(f), nn.ReLU(),
                      nn.MaxPool2d(2), nn.Dropout2d(p_descarte / 2)]
            entrada = f
        self.rasgos = nn.Sequential(*capas)
        self.clasificador = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(p_descarte),
            nn.Linear(entrada, n_clases),
        )

    def forward(self, x):
        return self.clasificador(self.rasgos(x))

## 4.3. Red densa simple

Corresponde al ejercicio 5. Aplana la imagen a un vector de 12,288 valores y la
pasa por dos capas ocultas. Se incluye para cuantificar cuanto aporta la
convolucion, porque al aplanar se pierde toda la relacion espacial entre
pixeles vecinos, que es precisamente lo que distingue una mano de otra.

In [4]:
class RedDensa(nn.Module):
    def __init__(self, n_clases=N_CLASES, ocultas=(512, 256), p_descarte=0.3):
        super().__init__()
        capas, entrada = [nn.Flatten()], 3 * LADO * LADO
        for h in ocultas:
            capas += [nn.Linear(entrada, h), nn.ReLU(), nn.Dropout(p_descarte)]
            entrada = h
        capas.append(nn.Linear(entrada, n_clases))
        self.red = nn.Sequential(*capas)

    def forward(self, x):
        return self.red(x)

## 4.4. Comparacion de tamano y verificacion

Se confirma que las tres redes producen la forma de salida correcta y se
compara su numero de parametros.

In [5]:
def n_parametros(modelo):
    return sum(p.numel() for p in modelo.parameters() if p.requires_grad)


lote_prueba = torch.randn(8, 3, LADO, LADO)
filas = []
for nombre, modelo in [("CNN A base", CNNBase()),
                       ("CNN B profunda", CNNProfunda()),
                       ("Red densa", RedDensa())]:
    with torch.no_grad():
        salida = modelo(lote_prueba)
    filas.append({
        "Modelo": nombre,
        "Parametros": n_parametros(modelo),
        "Salida": str(tuple(salida.shape)),
    })

tabla = pd.DataFrame(filas)
display(tabla.style.format({"Parametros": "{:,}"}))
print(f"todas las salidas deben ser (8, {N_CLASES})")

,Modelo,Parametros,Salida
0,CNN A base,"2,198,109","(8, 29)"
1,CNN B profunda,"1,181,629","(8, 29)"
2,Red densa,"6,430,749","(8, 29)"


todas las salidas deben ser (8, 29)


La red densa tiene mas parametros que las dos redes convolucionales juntas y
aun asi no puede ver que dos pixeles vecinos estan relacionados. Es el
argumento cuantitativo de por que se espera que pierda contra las CNN, y es lo
que el ejercicio 5 pide discutir.

## 4.5. Algoritmo clasico

Corresponde al ejercicio 6. Se selecciona **SVM con nucleo RBF sobre
componentes principales**, y se acompana de KNN como control.

El motivo de la seleccion es el siguiente. Un vector de pixeles tiene 12,288
dimensiones, y una SVM sobre esa cantidad de rasgos es lenta y propensa al
sobreajuste. Reducir con PCA a unos 150 componentes conserva la mayor parte de
la varianza, quita el ruido de fondo y deja el problema en un tamano donde el
nucleo RBF puede modelar fronteras curvas entre clases parecidas. Historicamente
esa combinacion fue la linea base fuerte de clasificacion de imagenes antes de
las redes profundas, asi que es la comparacion honesta contra una CNN.

KNN se incluye por una razon de diagnostico que sale del analisis exploratorio.
Si los cuadros de una misma clase son casi identicos, KNN deberia obtener una
exactitud altisima simplemente porque cada imagen de prueba tiene un vecino
casi igual en entrenamiento. Un KNN muy bueno seria entonces una senal de
redundancia y no de que el problema sea facil.

Random Forest se descarta como opcion principal porque decide sobre pixeles
individuales, y un pixel aislado a 64x64 casi no informa sobre la pose de la
mano.

In [6]:
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

modelo_svm = Pipeline([
    ("escala", StandardScaler()),
    ("pca", PCA(n_components=150, random_state=42)),
    ("svm", SVC(kernel="rbf", C=10, gamma="scale")),
])

modelo_knn = Pipeline([
    ("escala", StandardScaler()),
    ("pca", PCA(n_components=150, random_state=42)),
    ("knn", KNeighborsClassifier(n_neighbors=3)),
])

print("SVM:", " -> ".join(p[0] for p in modelo_svm.steps))
print("KNN:", " -> ".join(p[0] for p in modelo_knn.steps))

SVM: escala -> pca -> svm
KNN: escala -> pca -> knn


## 4.6. Rejilla de tuneo prevista

El tuneo se hace contra el conjunto de validacion, nunca contra el de prueba.
La rejilla queda definida aqui para que la entrega final solo tenga que
ejecutarla.

In [7]:
rejillas = {
    "CNN A base": {
        "filtros": ["(32,64,128)", "(64,128,256)"],
        "p_descarte": [0.25, 0.4],
        "lr": [1e-3, 5e-4],
    },
    "CNN B profunda": {
        "filtros": ["(32,64,128,256)", "(16,32,64,128)"],
        "p_descarte": [0.3, 0.5],
        "lr": [1e-3, 3e-4],
    },
    "Red densa": {
        "ocultas": ["(512,256)", "(1024,512,256)"],
        "p_descarte": [0.3, 0.5],
        "lr": [1e-3, 5e-4],
    },
    "SVM sobre PCA": {
        "n_components": [100, 150],
        "C": [1, 10],
        "gamma": ["scale", 0.01],
    },
}

filas_r = [{"Modelo": m, "Hiperparametro": h, "Valores": ", ".join(map(str, v))}
           for m, g in rejillas.items() for h, v in g.items()]
display(pd.DataFrame(filas_r))
print(f"combinaciones totales por modelo: "
      f"{[int(np.prod([len(v) for v in g.values()])) for g in rejillas.values()]}")

,Modelo,Hiperparametro,Valores
0,CNN A base,filtros,"(32,64,128), (64,128,256)"
1,CNN A base,p_descarte,"0.25, 0.4"
2,CNN A base,lr,"0.001, 0.0005"
3,CNN B profunda,filtros,"(32,64,128,256), (16,32,64,128)"
4,CNN B profunda,p_descarte,"0.3, 0.5"
5,CNN B profunda,lr,"0.001, 0.0003"
6,Red densa,ocultas,"(512,256), (1024,512,256)"
7,Red densa,p_descarte,"0.3, 0.5"
8,Red densa,lr,"0.001, 0.0005"
9,SVM sobre PCA,n_components,"100, 150"


combinaciones totales por modelo: [8, 8, 8, 8]


## 4.7. Protocolo de evaluacion

El criterio principal es la exactitud sobre el conjunto de prueba propio, con
el piso del azar en 3.4 por ciento porque hay 29 clases balanceadas.

Se acompana de tres cosas. La matriz de confusion, para verificar si los
errores caen donde el analisis exploratorio los predijo, es decir en el bloque
U, V y R. El puntaje F1 macro, que al estar balanceado el dataset deberia
parecerse a la exactitud y sirve de control. Y la exactitud sobre las fotos
propias del ejercicio 8, que es la unica medida que dice si el modelo sirve
fuera de la sesion de grabacion original.

Se usa parada temprana sobre la perdida de validacion con paciencia de cinco
epocas, para no premiar modelos que solo memorizan.

## 4.8. Verificacion de que el pipeline entrena

Se corren tres epocas de la CNN base. No es el entrenamiento final, solo la
comprobacion de que los datos, el modelo y el bucle de optimizacion funcionan
juntos y de que la perdida baja.

In [8]:
def entrenar_breve(modelo, epocas=3, lote=128, lr=1e-3):
    modelo = modelo.to(DISPOSITIVO)
    cargador = DataLoader(conjuntos["train"], batch_size=lote, shuffle=True)
    cargador_val = DataLoader(conjuntos["val"], batch_size=256)
    criterio = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(modelo.parameters(), lr=lr)

    historial = []
    for epoca in range(epocas):
        modelo.train()
        perdida_total = 0.0
        for xb, yb in cargador:
            xb, yb = xb.to(DISPOSITIVO), yb.to(DISPOSITIVO)
            opt.zero_grad()
            perdida = criterio(modelo(xb), yb)
            perdida.backward()
            opt.step()
            perdida_total += perdida.item() * len(yb)

        modelo.eval()
        correctos = 0
        with torch.no_grad():
            for xb, yb in cargador_val:
                xb, yb = xb.to(DISPOSITIVO), yb.to(DISPOSITIVO)
                correctos += (modelo(xb).argmax(1) == yb).sum().item()
        historial.append({
            "Epoca": epoca + 1,
            "Perdida entrenamiento": perdida_total / len(conjuntos["train"]),
            "Exactitud validacion": correctos / len(conjuntos["val"]),
        })
    return pd.DataFrame(historial)


t0 = time.time()
historial = entrenar_breve(CNNBase())
display(historial.style.format({"Perdida entrenamiento": "{:.4f}",
                                "Exactitud validacion": "{:.2%}"}))
print(f"tiempo {time.time()-t0:.0f}s en {DISPOSITIVO}")
print(f"piso del azar: {1/N_CLASES:.2%}")

,Epoca,Perdida entrenamiento,Exactitud validacion
0,1,2.4811,56.40%
1,2,1.0500,78.74%
2,3,0.6002,87.13%


tiempo 12s en mps
piso del azar: 3.45%


## 5. Plan de procesamiento de imagenes

Corresponde al ejercicio 7. El aumento de datos se define aqui y se aplica
sobre los modelos ya entrenados sin aumento, para poder medir la diferencia.

### Por que un giro horizontal cambia el significado

Un giro horizontal no es un aumento neutro en este problema. Todas las imagenes
del dataset son de la misma mano, asi que reflejar produce la mano contraria.
Eso trae dos consecuencias distintas.

La primera es que varias letras del alfabeto ASL se distinguen por hacia donde
apunta la mano y no solo por la forma de los dedos. G y H se hacen apuntando de
lado, y P y Q se diferencian de K y G basicamente por la orientacion. Al
reflejar, la direccion se invierte y la imagen se acerca visualmente a la clase
equivocada mientras conserva la etiqueta original. Se le estaria ensenando al
modelo que dos configuraciones opuestas son la misma letra.

La segunda es que, aun para las letras donde reflejar si produce una sena
valida hecha con la otra mano, se le pide al modelo que aprenda una invariancia
que el problema no necesita, porque tanto el entrenamiento como la prueba usan
la misma mano. Gasta capacidad en algo que no se le va a preguntar.

### Transformaciones que si tienen sentido

| Transformacion | Rango | Por que aplica |
|---|---|---|
| Rotacion pequena | mas menos 12 grados | La mano no siempre queda perfectamente vertical frente a la camara |
| Traslacion | hasta 10 por ciento | La mano no queda siempre centrada en el encuadre |
| Escala | 0.9 a 1.1 | Simula acercarse o alejarse de la camara |
| Brillo y contraste | mas menos 20 por ciento | Es la variacion mas realista al cambiar de cuarto o de luz |
| Desenfoque leve | radio hasta 1 | Simula el movimiento y el enfoque imperfecto de una camara en vivo |
| Borrado aleatorio | un recuadro pequeno | Obliga a no depender de una sola region de la imagen |

### Transformaciones que no se deben usar

| Transformacion | Por que no |
|---|---|
| Giro horizontal | Invierte la orientacion y confunde G, H, P y Q |
| Giro vertical | Produce poses que ninguna mano hace |
| Rotacion grande | Una rotacion fuerte convierte unas letras en otras |
| Deformacion fuerte | Cambia la geometria de los dedos, que es la senal que hay que aprender |
| Cambio de tono agresivo | Rompe el color de piel, que es lo que separa la mano del fondo |

La prioridad es el brillo y el contraste, porque el analisis exploratorio
mostro que la iluminacion del dataset es muy uniforme, entre 118.8 y 141.4 de
brillo promedio, mientras que las fotos propias del ejercicio 8 van a tener
condiciones mucho mas variadas. Ese es el desajuste mas probable entre el
entrenamiento y el uso real.

## 6. Lo que sigue para la entrega final

1. Ejecutar la rejilla de tuneo de las dos CNN y quedarse con la mejor por
   exactitud de validacion.
2. Entrenar la red densa y la SVM sobre PCA con su propio tuneo.
3. Comparar los cuatro modelos sobre el mismo conjunto de prueba y revisar la
   matriz de confusion contra los pares que predijo el analisis exploratorio.
4. Reentrenar el mejor modelo con aumento de datos y medir la diferencia.
5. Tomar las fotos propias, minimo cinco letras por integrante, guardarlas en
   `data/team-dataset/` y evaluar el mejor modelo sobre ellas.
6. Escribir la reflexion de accesibilidad y sesgo. El punto de partida ya esta
   en los hallazgos del analisis exploratorio, una sola persona, un solo tono
   de piel, un solo fondo y una sola condicion de iluminacion.

# 9. Reflexiones de accesibilidad y sesgo

Las limitantes del dataset incluyen que algunas letras conllevan movimiento tal como el caso de las letras J y Z, además de que es un dataset con una escalada de datos considerada pequeña en comparación con otros. Además, la falta de variedad de tonos de piel de los participantes, edades (influyendo el tamaño y articulación de las manos) y ambiente controlado pueden afectar la exactitud del modelo en contextos de la vida real.
Haría falta resolver el estudio de letras y señas que representan palabras o contextos para dar una base de datos más amplia y realista, incluyendo por supuesto una mayor variedad en los sujetos de prueba.